# B08-03 — 회고적 최적화 성과·선택 편향 감사

**BioCulture · 실제 E. coli 배지 데이터 · 교육용 Scientific AI**

B08-01/B08-02가 저장한 순차 선택 기록을 별도 방식으로 감사한다. **새 배양 실험을 수행하거나 새로운 BO 모델을 학습하지 않는다.**

학습 목표: (1) 고정 예산별 발견 성과와 초기값 대비 개선량, (2) 같은 초기 시드를 사용하는 전략의 짝지은 비교, (3) 선택된 조성의 성분·실험 라운드 편향, (4) 유효한 목표값만 남긴 후보군의 한계, (5) 기존 결과의 재현성과 허용 주장 범위 구분.

**주의:** 예산 `step=0`은 B08-02의 이력 CSV에 없으며 `initial_best`로 복원한다. `oracle_max`는 전체 역사적 실측값으로 계산하는 **사후 참조 상한**으로, 실제 선택 전략에 제공하지 않는다. 회고적 분석에서 후보 전체의 성분 농도는 이미 알려졌다고 가정하지만 숨긴 목표값은 선택 전에 사용할 수 없다.

## 1. 프로젝트 파일과 환경 점검

입력 파일은 B08-00의 실제 조성표, B07-00의 성분 목록, B08-02의 선택 이력·요약, 가능하면 B08-01의 과거 결과이다. 파일이 누락되면 임의 값을 생성하지 않고 중단한다.

In [ ]:
from pathlib import Path
import json, hashlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd=Path.cwd().resolve()
ROOT=next((p for p in [cwd,*cwd.parents] if (p/'results'/'bioculture'/'B08_02_bayesian_optimization_v1').is_dir()),None)
if ROOT is None: raise FileNotFoundError('B08-02 결과가 포함된 bioimage-ai 프로젝트 안에서 실행하세요.')
BASE=ROOT/'results'/'bioculture'
CAND=BASE/'B08_00_candidate_design_v2'/'candidate_pool_measured_profiles.csv'
FEATURE=BASE/'B07_00_ml_data_audit_v2'/'medium_feature_audit.csv'
TR=BASE/'B08_02_bayesian_optimization_v1'/'bayesian_sequential_selection_trace.csv'
SUM=BASE/'B08_02_bayesian_optimization_v1'/'bayesian_sequential_selection_summary.csv'
PRIOR=BASE/'B08_01_sequential_selection_v1'/'sequential_selection_summary.csv'
OUT=BASE/'B08_03_selection_bias_audit_v1';OUT.mkdir(parents=True,exist_ok=True)
for p in [CAND,FEATURE,TR,SUM]:
    if not p.is_file(): raise FileNotFoundError(p)
TARGETS=['K_median','r_median']; STRATEGIES=['Random','RF_greedy','GP_EI']
SEEDS=[17,43,101];BUDGETS=[0,1,5,10,15,20];N_INITIAL=30;N_STEPS=20
print('Python:',sys.version.split()[0]);print('분석 폴더:',OUT)


## 2. 자료의 식별자·자료형·이력 불변조건 감사

같은 목표·시드에 세 전략의 초기 최고값이 반드시 같아야 하며, 이미 선택된 조성은 다시 선택할 수 없다. 각각의 `revealed_value`를 원본 후보군과 일대일 대조한다. 이 대조는 사후 감사에서만 수행된다.

In [ ]:
candidate=pd.read_csv(CAND,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
fi=pd.read_csv(FEATURE,encoding='utf-8-sig');features=fi['feature'].astype(str).tolist()
trace=pd.read_csv(TR,encoding='utf-8-sig',dtype={'chosen_medium_profile_id':'string'})
summary=pd.read_csv(SUM,encoding='utf-8-sig')
assert len(candidate)==1025 and candidate.medium_profile_id.is_unique
assert len(features)==44 and set(features).issubset(candidate.columns)
assert set(TARGETS).issubset(candidate.columns)
for target in TARGETS: candidate[target]=pd.to_numeric(candidate[target],errors='coerce')
for col in ['step','seed','initial_best','best_so_far','revealed_value','n_revealed']:
    trace[col]=pd.to_numeric(trace[col],errors='raise')
expected_runs=pd.MultiIndex.from_product([TARGETS,SEEDS,STRATEGIES],names=['target','seed','strategy'])
actual_runs=pd.MultiIndex.from_frame(trace[['target','seed','strategy']].drop_duplicates())
assert len(actual_runs)==len(expected_runs) and set(actual_runs)==set(expected_runs)
assert len(trace)==len(expected_runs)*N_STEPS==360
assert not trace[['target','seed','strategy','step']].duplicated().any()
assert trace.groupby(['target','seed','strategy']).step.apply(lambda s:set(s)==set(range(1,N_STEPS+1))).all()
assert trace.groupby(['target','seed','strategy']).chosen_medium_profile_id.nunique().eq(N_STEPS).all()
assert trace.groupby(['target','seed']).initial_best.nunique().eq(1).all()
assert (trace.n_revealed==N_INITIAL+trace.step).all()
assert trace.groupby(['target','seed','strategy']).best_so_far.apply(lambda s:(s.diff().dropna()>=-1e-10).all()).all()
for target in TARGETS:
    known=candidate.set_index('medium_profile_id')[target]
    t=trace.loc[trace.target==target].copy()
    vals=t.chosen_medium_profile_id.map(known)
    assert vals.notna().all() and np.allclose(vals,t.revealed_value,atol=1e-9)
    # 누적 최대값이 단계별로 실제로 갱신됐는지 다시 계산
    for _,g in t.groupby(['seed','strategy']):
        g=g.sort_values('step');calc=np.maximum.accumulate(np.r_[g.initial_best.iloc[0],g.revealed_value.to_numpy()])[1:]
        assert np.allclose(calc,g.best_so_far.to_numpy(),atol=1e-9)
assert len(summary)==18
print('검증된 실행:',len(actual_runs),'건 / 선택 기록:',len(trace),'건')
print('입력 조건:',len(candidate),'개 / 성분:',len(features),'개')


## 3. 공통 예산에 따른 발견 성과와 사후 참조 상한

`best_so_far`는 현재까지 확인한 최고값이다. `step=0`에서 시작해 1·5·10·15·20회의 성과를 비교한다. **전체 후보의 관측 최대값은 참조치**이지, 실험 중 공개되는 정보가 아니다.

In [ ]:
curve_rows=[]; budget_rows=[]
for (target,seed,strategy),g in trace.groupby(['target','seed','strategy'],sort=True):
    g=g.sort_values('step')
    origin=float(g.initial_best.iloc[0]);oracle_max=float(candidate[target].max())
    grid=pd.concat([pd.DataFrame([{'step':0,'best_so_far':origin}]),g[['step','best_so_far']]],ignore_index=True)
    for _,row in grid.iterrows():
        step=int(row.step);best=float(row.best_so_far)
        curve_rows.append(dict(target=target,seed=int(seed),strategy=strategy,step=step,best_so_far=best,
                               gain_from_initial=best-origin,remaining_gap_to_historical_best=oracle_max-best,
                               historical_global_best=oracle_max))
        if step in BUDGETS:budget_rows.append(curve_rows[-1].copy())
curves=pd.DataFrame(curve_rows);budgets=pd.DataFrame(budget_rows)
assert len(curves)==18*(N_STEPS+1) and len(budgets)==18*len(BUDGETS)
final=budgets[budgets.step==N_STEPS].copy()
final['mean_best_over_steps_recomputed']=curves.loc[curves.step>0].groupby(['target','seed','strategy']).best_so_far.transform('mean').loc[curves.step==N_STEPS].to_numpy()
q=final.merge(summary,on=['target','seed','strategy'],suffixes=('_audit','_source'),validate='one_to_one')
assert np.allclose(q.best_so_far,q.final_best,atol=1e-9)
assert np.allclose(q.gain_from_initial_audit,q.gain_from_initial_source,atol=1e-9)
print('예산 20회 최종 발견값의 평균 (같은 데이터 내 시드 3개):')
print(final.groupby(['target','strategy'])[['best_so_far','gain_from_initial']].mean().round(5).to_string())
print('사후 역사적 최고값:',{t:round(float(candidate[t].max()),5) for t in TARGETS})


## 4. 시드별 짝지은 비교와 발견 속도

세 전략은 같은 목표·시드에서 **동일한 초기 30개 조성**을 공유한다. 따라서 전략 간 성과 차이는 시드를 맞추어 계산한다. `AUC`는 1~20회 `best_so_far`의 평균으로 정의하며, 이 연구에서는 새 생물학적 반복에 대한 신뢰구간을 산출하지 않는다.

In [ ]:
paired=[]
for (target,seed),g in final.groupby(['target','seed']):
    d=g.set_index('strategy')
    for left,right in [('RF_greedy','Random'),('GP_EI','Random'),('GP_EI','RF_greedy')]:
        paired.append(dict(target=target,seed=int(seed),strategy_a=left,strategy_b=right,
                           delta_final_best=float(d.loc[left,'best_so_far']-d.loc[right,'best_so_far']),
                           delta_gain=float(d.loc[left,'gain_from_initial']-d.loc[right,'gain_from_initial'])))
paired=pd.DataFrame(paired)
auc=(curves[curves.step>0].groupby(['target','seed','strategy'],as_index=False)
     .agg(mean_best_over_20_steps=('best_so_far','mean'),n_steps=('step','size')))
assert auc.n_steps.eq(20).all()
print('짝지은 시드 비교:')
print(paired.to_string(index=False,float_format=lambda x:f'{x:.4f}'))
print('\n20단계 평균 최고값:')
print(auc.groupby(['target','strategy']).mean_best_over_20_steps.mean().round(4).to_string())


## 5. 선택 편향: 어떤 조성이 자주 선택되는가?

사후 감사에서 선택된 조성과 초기 후보군 전체의 성분 분포를 비교한다. 분석 목적은 **선택 알고리즘이 어디를 탐색했는지** 확인하는 것이다. 같은 시드에서 여러 차례 등장하는 조성이 있으므로 선택 행은 독립 표본이 아니다.

성분별 `standardized mean difference` (SMD)는 `(선택된 조성 평균 − 전체 유효 후보 평균) / 전체 유효 후보 표준편차`로 계산한다. SMD는 인과효과가 아니다.

In [ ]:
selected=trace[['target','seed','strategy','chosen_medium_profile_id','step']].copy()
selected=selected.merge(candidate[['medium_profile_id',*features,'n_rounds']].rename(columns={'medium_profile_id':'chosen_medium_profile_id'}),on='chosen_medium_profile_id',how='left',validate='many_to_one')
assert selected[features].notna().all().all()
bias=[]
for target in TARGETS:
    universe=candidate.loc[candidate[target].notna(),features].astype(float)
    sd=universe.std(ddof=0).replace(0,np.nan)
    for strategy in STRATEGIES:
        pick=selected[(selected.target==target)&(selected.strategy==strategy)]
        for feature in features:
            a=float(universe[feature].mean());b=float(pick[feature].mean());s=float(sd[feature])
            bias.append(dict(target=target,strategy=strategy,feature=feature,
                             universe_mean=a,selected_mean=b,standardized_mean_difference=(b-a)/s if s>0 else np.nan,
                             n_selected_records=len(pick)))
bias=pd.DataFrame(bias)
round_bias=(selected.groupby(['target','strategy'],as_index=False).agg(
  n_selection_records=('step','size'),fraction_multi_round=('n_rounds',lambda x:float((x>1).mean())),
  unique_selected_profiles=('chosen_medium_profile_id','nunique')))
# 서로 다른 seed가 동일 조성을 재선택할 수 있으므로 unique가 60 이하일 수 있다.
print('절대 SMD 상위 12개:')
print(bias.assign(abs_smd=lambda x:x.standardized_mean_difference.abs()).nlargest(12,'abs_smd')[['target','strategy','feature','standardized_mean_difference']].to_string(index=False))
print('\n선택 후보 반복 라운드 구성:')
print(round_bias.to_string(index=False))


## 6. 후보 자격 필터와 실험 라운드 교란 한계

B08-02는 목표별 유효 `K` 또는 `r`이 존재하는 조성만 후보로 사용했다. 결과 결측이 선택 전 이미 알려져 있는 것처럼 필터링한 **회고적 선택 가능성 편향**을 구분한다. 전체 후보군 중 복수 라운드에 등장한 조성이 적다는 점도 별도로 기록한다.

In [ ]:
eligibility=[]
for target in TARGETS:
    v=candidate[target].notna()
    eligibility.append({'target':target,'all_measured_profiles':len(candidate),
                        'eligible_known_outcome':int(v.sum()),'excluded_missing_outcome':int((~v).sum()),
                        'eligible_multi_round_profiles':int(((candidate.n_rounds>1)&v).sum())})
eligibility=pd.DataFrame(eligibility)
assert int((candidate.n_rounds>1).sum())==25
print(eligibility.to_string(index=False))
print('주의: 목표값이 결측인 조성은 사전에 제외됨. 완전한 전향적 실험 비용·실패율 평가가 아님.')


## 7. 시각화 및 원본 결과 대조

시드 평균 발견 곡선과 선택 성분의 편향을 표시한다. 아래 시각화는 새로운 생물학적 실험 결과가 아니라 저장된 회고적 이력의 재표현이다.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.2),constrained_layout=True)
for ax,t in zip(axs,TARGETS):
    for strategy in STRATEGIES:
        s=curves[(curves.target==t)&(curves.strategy==strategy)]
        v=s.groupby('step').best_so_far.agg(['mean','min','max'])
        ax.plot(v.index,v['mean'],label=strategy,marker='.',markersize=3)
        ax.fill_between(v.index,v['min'],v['max'],alpha=0.12)
    ax.set(title=f'{t} retrospective discovery',xlabel='Additional queries',ylabel='Best observed outcome');ax.legend()
p_fig=OUT/'budget_discovery_curves.png';fig.savefig(p_fig,dpi=160);plt.show();plt.close(fig)
if PRIOR.is_file():
    old=pd.read_csv(PRIOR,encoding='utf-8-sig')
    v=summary[summary.strategy.isin(['Random','RF_greedy'])].merge(old,on=['target','seed','strategy'],suffixes=('_new','_old'),validate='one_to_one')
    assert len(v)==12
    for col in ['initial_best','final_best','gain_from_initial']:
        assert np.allclose(v[f'{col}_new'],v[f'{col}_old'],atol=1e-9)
    print('B08-01 대조: 12개 실행 정합성 확인')
else:
    print('B08-01 이전 요약 파일 없음: 독립 대조 생략')


## 8. 결과 저장·불변조건 검사·과학적 결론의 범위

원본 실측값이 있는 후보만 사용했고, 전체 사후 최고값은 평가 목적으로만 참조하였다. 후보의 **실측값 격리를 유지한 원래 선택 알고리즘의 구조는 B08-01·02**가 담당한다. 이 Notebook은 저장 이력에 대한 사후 감사이다.

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

outputs={
 'budget_per_run_metrics.csv':budgets,
 'paired_seed_strategy_differences.csv':paired,
 'mean_best_over_steps.csv':auc,
 'selected_feature_bias_audit.csv':bias,
 'selected_round_bias_audit.csv':round_bias,
 'candidate_eligibility_audit.csv':eligibility,
 'discovery_curves_with_initial.csv':curves,
}
for name,data in outputs.items():
    dest=OUT/name;data.to_csv(dest,index=False,encoding='utf-8-sig')
    test=pd.read_csv(dest,encoding='utf-8-sig')
    assert len(test)==len(data) and list(test.columns)==list(data.columns),name
report={
 'status':'RETROSPECTIVE_SELECTION_AUDIT_COMPLETED_NOT_PROSPECTIVE_VALIDATION',
 'n_profiles':len(candidate),'n_features':len(features),'n_trace_records':len(trace),
 'n_runs':len(actual_runs),'n_initial':N_INITIAL,'budgets':BUDGETS,
 'seeds':SEEDS,'targets':TARGETS,'strategies':STRATEGIES,
 'historical_max_for_posthoc_audit_only':True,
 'source_trace_values_rechecked_against_candidate_table':True,
 'selection_outcome_filter_used_before_simulation':True,
 'round_shift_not_solved':True,'n_multi_round_profiles':int((candidate.n_rounds>1).sum()),
 'claims_not_supported':['new biological experiments','algorithm superiority across laboratories',
                         'calibrated uncertainty','causal ingredient effects','failure cost modeled'],
}
rp=OUT/'B08_03_validation_report.json';rp.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
manifest_inputs=[CAND,FEATURE,TR,SUM,*([PRIOR] if PRIOR.is_file() else [])]
manifest_outputs=[OUT/k for k in outputs]+[p_fig,rp]
manifest=pd.DataFrame([{'path':str(p.relative_to(ROOT)),'role':'input' if p in manifest_inputs else 'output',
                        'bytes':p.stat().st_size,'sha256':sha256(p)} for p in manifest_inputs+manifest_outputs])
manifest.to_csv(OUT/'input_output_sha256_manifest.csv',index=False,encoding='utf-8-sig')
print('검증 상태:',report['status']);print('결과 파일:',len(outputs)+3,'개');print('위치:',OUT)


## 연습문제 — 정답·Python 풀이는 별도 Word 해설서에 수록

1. **예산 민감도:** 5회와 20회 선택에서 세 전략의 평균 최고 발견값을 비교하고 어떤 전략의 개선 폭이 큰지 계산하라.
2. **짝지은 비교:** 같은 목표·시드에서 `GP_EI - RF_greedy`의 최종 최고값 차이를 모두 계산하고 평균의 한계를 설명하라.
3. **선택 편향:** `K_median`에 대해 선택된 배지 중 Glucose의 SMD를 전략별로 비교하고 왜 인과효과가 아닌지 설명하라.
4. **회고적 유효 후보 편향:** 목표별 결측 후보 수를 계산하고 전향적 실험 평가가 어려운 이유를 설명하라.
5. **검증 강화:** 라운드가 다른 자료로 일반화하려면 어떤 독립 실험 설계와 검증 지표가 필요한지 제안하라.

**해석상의 주의:** 시드 3개는 독립적 생물학적 반복이 아니다. 역사적 최고값은 사후 참조치이다. 성분 중요도는 실험적 인과효과가 아니다.
